# Mão na Roda — Fine-tuning do BERTimbau (split 70/15/15)

Treina o classificador de falhas automotivas (10 classes) sobre a base de **2800 relatos**, com split estratificado **70% treino / 15% validação / 15% teste**.

Esquema: `neuralmind/bert-base-portuguese-cased`, taxa de aprendizado 5×10⁻⁵, até 20 épocas com early stopping (paciência 5, monitorando F1-macro de validação), warmup linear de 5% dos passos, 3 sementes (42, 123, 2024), e baseline TF-IDF + regressão logística para comparação.

## 1. Instalação e imports

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy torch

In [ ]:
import random

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

## 2. Carregar os dados

Ajuste o caminho do CSV e os nomes das colunas se forem diferentes. É esperado um CSV com uma coluna de texto (`relato`) e uma coluna de classe (`classe`).

In [ ]:
CSV_PATH = "relatos.csv"  # ajuste para o caminho real do seu arquivo
TEXT_COL = "relato"
LABEL_COL = "classe"

df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
df = df.rename(columns={TEXT_COL: "relato", LABEL_COL: "classe"})

print("Total de relatos:", len(df))
print(df["classe"].value_counts())

In [ ]:
label_list = sorted(df["classe"].unique())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
df["label"] = df["classe"].map(label2id)

NUM_LABELS = len(label_list)
print("Classes:", label_list)

## 3. Funções auxiliares (split, tokenização, treino, baseline)

In [ ]:
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MAX_LENGTH = 256
SEEDS = (42, 123, 2024)

TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 0.70, 0.15, 0.15
SPLIT_NAME = "70_15_15"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def split_stratified(df, train_size, val_size, test_size, seed):
    assert abs(train_size + val_size + test_size - 1.0) < 1e-6, "as proporções devem somar 1.0"
    train_df, temp_df = train_test_split(
        df, train_size=train_size, stratify=df["label"], random_state=seed
    )
    relative_val = val_size / (val_size + test_size)
    val_df, test_df = train_test_split(
        temp_df, train_size=relative_val, stratify=temp_df["label"], random_state=seed
    )
    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


def tokenize_df(df):
    ds = Dataset.from_pandas(df[["relato", "label"]])
    ds = ds.map(
        lambda x: tokenizer(x["relato"], truncation=True, padding="max_length", max_length=MAX_LENGTH),
        batched=True,
    )
    ds = ds.remove_columns(["relato"])
    ds.set_format("torch")
    return ds


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

In [ ]:
def train_bertimbau(train_df, val_df, test_df, seed, output_dir):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    train_ds = tokenize_df(train_df)
    val_ds = tokenize_df(val_df)
    test_ds = tokenize_df(test_df)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
    )

    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=20,
        warmup_ratio=0.05,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        seed=seed,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
    )

    trainer.train()
    test_metrics = trainer.evaluate(test_ds)
    return trainer, test_metrics


def train_baseline(train_df, test_df, seed):
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    X_train = vectorizer.fit_transform(train_df["relato"])
    X_test = vectorizer.transform(test_df["relato"])

    clf = LogisticRegression(max_iter=1000, random_state=seed)
    clf.fit(X_train, train_df["label"])
    preds = clf.predict(X_test)

    return {
        "accuracy": accuracy_score(test_df["label"], preds),
        "f1_macro": f1_score(test_df["label"], preds, average="macro"),
    }

## 4. Treinamento — 3 sementes

In [ ]:
rows = []
for seed in SEEDS:
    print(f"\n=== Split {SPLIT_NAME} | seed {seed} ===")
    train_df, val_df, test_df = split_stratified(df, TRAIN_SIZE, VAL_SIZE, TEST_SIZE, seed)
    print(f"treino={len(train_df)}  validação={len(val_df)}  teste={len(test_df)}")

    _, bert_metrics = train_bertimbau(
        train_df, val_df, test_df, seed, output_dir=f"./results_{SPLIT_NAME}_seed{seed}"
    )
    baseline_metrics = train_baseline(train_df, test_df, seed)

    rows.append({
        "seed": seed,
        "bert_accuracy": bert_metrics["eval_accuracy"],
        "bert_f1_macro": bert_metrics["eval_f1_macro"],
        "baseline_accuracy": baseline_metrics["accuracy"],
        "baseline_f1_macro": baseline_metrics["f1_macro"],
    })

results_70_15_15 = pd.DataFrame(rows)
results_70_15_15

## 5. Resumo (média e desvio padrão entre as 3 sementes)

In [ ]:
summary = results_70_15_15[
    ["bert_accuracy", "bert_f1_macro", "baseline_accuracy", "baseline_f1_macro"]
].agg(["mean", "std"])
summary

In [ ]:
best_row = results_70_15_15.sort_values("bert_f1_macro", ascending=False).iloc[0]
print("Melhor semente (maior F1-macro no teste):")
print(best_row)

## 6. Salvar o melhor modelo desta rodada

In [ ]:
BEST_SEED = int(best_row["seed"])

train_df, val_df, test_df = split_stratified(df, TRAIN_SIZE, VAL_SIZE, TEST_SIZE, seed=BEST_SEED)
best_trainer, best_metrics = train_bertimbau(
    train_df, val_df, test_df, BEST_SEED, output_dir="./best_model_tmp_70_15_15"
)

best_trainer.save_model("./mao_na_roda_bertimbau_70_15_15")
tokenizer.save_pretrained("./mao_na_roda_bertimbau_70_15_15")

print("Modelo final salvo em ./mao_na_roda_bertimbau_70_15_15")
print(best_metrics)